# Research assistant: prototype

This is the **messy prototype** the session starts from. It works on the machine it was written on.
Read it, run it, and list what would break when someone else (or a server) runs it.

Kernel: `venvAO01` (see the session README). Keys are read from `../.env`; you never paste a key here.

In [1]:
# !pip install langchain langchain-openai tavily-python   # no versions: whatever is newest today
import os, time
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from tavily import TavilyClient

load_dotenv("../.env")
# OPENAI_API_KEY = "<paste-your-key-here>"   # the usual shortcut: a key in code ends up in git history
for key in ("OPENAI_API_KEY", "TAVILY_API_KEY"):
    if not os.getenv(key):
        raise RuntimeError(f"{key} is missing: copy .env.example to .env in the session folder and fill it in")

In [2]:
MODEL = "gpt-6-luna"   # hardcoded here; nothing records which model answered
SYSTEM_PROMPT = "You are a research assistant. Answer in 3 short bullet points with numbers and sources."

In [3]:
@tool
def web_search(query: str) -> list[dict]:
    """Search the web."""
    tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
    return tavily.search(query, max_results=3)["results"]

@tool
def calculate(expression: str) -> str:
    """Evaluate a math expression."""
    return str(eval(expression))   # runs ANY Python the model sends

In [6]:
question = "What seasonal COP do air-source heat pumps typically reach in cold climates, "\
           "and how many kWh of electricity does one need to deliver 12,000 kWh of heat at that COP?"
agent = create_agent(init_chat_model(MODEL, model_provider="openai"), tools=[web_search, calculate],
                     system_prompt=SYSTEM_PROMPT)
start = time.time()
result = agent.invoke({"messages": [{"role": "user", "content": question}]})
print(result["messages"][-1].text)
print("\nserved model:", result["messages"][-1].response_metadata["model_name"], f"| {time.time() - start:.1f}s")

Cold-climate air-source heat pumps typically achieve a seasonal COP of about 2–3; at a representative COP of 2.5, delivering 12,000 kWh of heat requires about **4,800 kWh of electricity** (12,000 ÷ 2.5).

served model: gpt-6-luna | 7.4s


## Scratch: trying a shorter answer

Someone tried a different prompt here, then went back and re-ran the cell above.

In [5]:
SYSTEM_PROMPT = "You are a research assistant. Reply in one sentence."